<a href="https://colab.research.google.com/github/Abhinay-880/Band-Name-Generator/blob/main/exercises/machine-learning/neural-networks/multilayer_perceptrons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multilayer Perceptrons
You should build an end-to-end machine learning pipeline using a multilayer perceptron model. In particular, you should do the following:
- Load the `mnist` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Split the dataset into training and test sets using [Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).
- Build an end-to-end machine learning pipeline, including a [multilayer perceptron](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) model.
- Optimize your pipeline by validating your design decisions.
- Test the best pipeline on the test set and report various [evaluation metrics](https://scikit-learn.org/0.15/modules/model_evaluation.html).  
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [11]:
import pandas as pd

# Load the local MNIST dataset from the datasets folder as specified
df = pd.read_csv('mnist.csv')
df.head()

,id,class,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,31953,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,34452,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,60897,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,36953,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1981,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
df.shape

(4000, 786)

In [13]:
df.isnull().sum().sum()

np.int64(0)

In [14]:
# Drop the 'id' column as it is not a feature, and separate 'class' as the label
X = df.drop(columns=['id', 'class'])
y = df['class']

# Split into training and test sets (e.g., 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (3200, 784)
X_test shape: (800, 784)


### 1. Scale Features
MLPs are sensitive to feature scaling. We'll divide our features by 255.0 to scale them to the range `[0, 1]`.

In [15]:
# Scale the pixel intensity values to [0, 1]
X_train_scaled = X_train / 255.0
X_test_scaled = X_test / 255.0

### 2. Build and Optimize MLP Classifier
Let's configure, train, and validate a Scikit-Learn `MLPClassifier`. We will set hyperparameters like `hidden_layer_sizes`, `activation`, `solver`, and `max_iter`.

In [16]:
# Initialize the Multilayer Perceptron model
# We'll use a single hidden layer of 100 neurons, 'adam' solver, and early stopping to prevent overfitting.
mlp = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42,
    early_stopping=True
)

# Train the model
mlp.fit(X_train_scaled, y_train)
print("Model training complete!")
print(f"Best validation score during training: {mlp.best_validation_score_:.4f}")

Model training complete!
Best validation score during training: 0.9094


### 3. Evaluate Pipeline on Test Set
Now we'll evaluate the performance of our optimized MLP classifier on the unseen test set using accuracy, a classification report, and a confusion matrix.

In [17]:
# Predictions on test set
y_pred = mlp.predict(X_test_scaled)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Set Accuracy: {accuracy:.4%}\n")

# Detailed Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Set Accuracy: 90.8750%

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99        70
           1       0.94      0.97      0.96       100
           2       0.80      0.82      0.81        73
           3       0.91      0.87      0.89        86
           4       0.90      0.91      0.91        80
           5       0.85      0.94      0.89        64
           6       0.97      0.92      0.94        90
           7       0.94      0.93      0.93        67
           8       0.89      0.86      0.88        94
           9       0.89      0.88      0.89        76

    accuracy                           0.91       800
   macro avg       0.91      0.91      0.91       800
weighted avg       0.91      0.91      0.91       800

Confusion Matrix:
[[69  0  0  0  0  1  0  0  0  0]
 [ 0 97  1  1  0  0  0  0  1  0]
 [ 1  2 60  2  3  0  2  1  2  0]
 [ 0  0  2 75  1  6  0  0  1  1]
 [ 0  1  1  0 73  0  1  0  2  2]
 [ 0  

### 4. Hyperparameter Tuning and Execution Time Tracking
We will define a grid of hyperparameters (e.g., different hidden layer configurations and learning rate values), run them sequentially, and log both the training time and the best validation score achieved during training.

In [18]:
import time
import pandas as pd
from sklearn.neural_network import MLPClassifier

# Define the hyperparameter grid
hidden_layer_options = [(50,), (100,), (100, 50)]
learning_rate_options = ['constant', 'adaptive']

results = []

print("Starting Hyperparameter Tuning...")
print("-" * 80)

for layers in hidden_layer_options:
    for lr_type in learning_rate_options:
        # Set up the MLP with specific hyperparameters
        mlp_tune = MLPClassifier(
            hidden_layer_sizes=layers,
            learning_rate=lr_type,
            activation='relu',
            solver='adam',
            max_iter=300,
            random_state=42,
            early_stopping=True
        )

        # Measure training duration
        start_time = time.time()
        mlp_tune.fit(X_train_scaled, y_train)
        end_time = time.time()

        elapsed_time = end_time - start_time
        best_val_score = mlp_tune.best_validation_score_

        # Save logs
        results.append({
            'Hidden Layers': str(layers),
            'Learning Rate Mode': lr_type,
            'Training Time (s)': round(elapsed_time, 3),
            'Validation Score': round(best_val_score, 4)
        })

        print(f"Layers: {layers} | LR Mode: {lr_type} | Time: {elapsed_time:.2f}s | Val Score: {best_val_score:.4f}")

# Convert results to DataFrame for comparison
results_df = pd.DataFrame(results)
display(results_df)

Starting Hyperparameter Tuning...
--------------------------------------------------------------------------------
Layers: (50,) | LR Mode: constant | Time: 1.64s | Val Score: 0.8906
Layers: (50,) | LR Mode: adaptive | Time: 1.61s | Val Score: 0.8906
Layers: (100,) | LR Mode: constant | Time: 2.75s | Val Score: 0.9094
Layers: (100,) | LR Mode: adaptive | Time: 4.75s | Val Score: 0.9094
Layers: (100, 50) | LR Mode: constant | Time: 4.09s | Val Score: 0.8938
Layers: (100, 50) | LR Mode: adaptive | Time: 3.55s | Val Score: 0.8938


,Hidden Layers,Learning Rate Mode,Training Time (s),Validation Score
0,"(50,)",constant,1.639,0.8906
1,"(50,)",adaptive,1.610,0.8906
2,"(100,)",constant,2.747,0.9094
3,"(100,)",adaptive,4.749,0.9094
4,"(100, 50)",constant,4.094,0.8938
5,"(100, 50)",adaptive,3.549,0.8938
